# Pokemon pure-RL plateau audit

## tl;dr

The run is learning to reproduce its own action distribution, not getting stronger against fixed opponents. Mean fixed held-out win rate moved from **11.59%** in iterations 0–19 to **10.55%** in iterations 162–181, while own-action accuracy rose from **48.75%** to **98.66%**. The current promotion gate can therefore keep advancing near-identical checkpoints without proving absolute improvement.

A specialist experiment is justified, but **Lucario is not the evidence-backed first deck**. Its recent reconstructed held-out rate is 4.38%, versus 25.00% for Crustle and 23.44% for Cornerstone Ogerpon. The July 12 top-ladder snapshot also gives Lucario only 2.1% game share and 35.5% win rate.

## Context & Methods

### Key assumptions

- Run metrics are a frozen snapshot of Inzi v12 through committed iteration 181, queried on July 19, 2026.
- `fixed held-out win rate` is the 200-game greedy evaluation against the four official bots, not the candidate-versus-incumbent promotion result.
- Deck-level held-out rates are reconstructed from `job_index` because `heldout_rows` drops `our_deck` and `archetype`. The formula mirrors the deterministic scheduler in `scripts/train_pure_rl.py`.
- Ladder frequency and ladder win rate come from `data/training_mixes/top_ladder.v1.json`, a top-cohort conditional snapshot rather than an unbiased census.
- The companion script `docs/pokemon_rl_plateau_audit.py` contains all calculations and assertions.

## Data

In [1]:
import runpy

audit = runpy.run_path("docs/pokemon_rl_plateau_audit.py")
results = audit["build_results"]()
audit["validate"](results)
run = results["run_snapshot"]
print(
    f"iterations={run['committed_iterations']} "
    f"scheduled_games={run['scheduled_source_games']} "
    f"heldout_first20={run['heldout_first_20']:.2%} "
    f"heldout_last20={run['heldout_last_20']:.2%} "
    f"change={run['heldout_change_percentage_points']:.2f}pp"
)

iterations=182 scheduled_games=372736 heldout_first20=11.59% heldout_last20=10.55% change=-1.04pp


## Results

In [2]:
selected_ids = ["crustle", "cornerstone-ogerpon", "rockets-mewtwo", "lucario"]
by_deck = {row["deck_id"]: row for row in results["deck_rows"]}
print("deck                   first20   last20  change    ladder share  ladder WR")
for deck_id in selected_ids:
    row = by_deck[deck_id]
    print(
        f"{deck_id:23} {row['heldout_first_20']:8.2%} {row['heldout_last_20']:9.2%} "
        f"{100 * row['heldout_change']:+7.2f}pp {row['ladder_game_share']:12.2%} "
        f"{row['ladder_win_rate']:11.2%}"
    )

deck                   first20   last20  change    ladder share  ladder WR
crustle                   20.31%    25.00%   +4.69pp       42.80%      51.20%
cornerstone-ogerpon       21.56%    23.44%   +1.88pp        9.50%      46.00%
rockets-mewtwo            14.69%    14.38%   -0.31pp        9.00%      63.20%
lucario                   12.50%     4.38%   -8.12pp        2.10%      35.50%


In [3]:
counts = results["balanced_eval_game_counts_by_deck_index"]
print(counts)
print(f"first eight decks receive {counts[0] / counts[-1]:.1f}x the games of the remaining nine")

[16, 16, 16, 16, 16, 16, 16, 16, 8, 8, 8, 8, 8, 8, 8, 8, 8]
first eight decks receive 2.0x the games of the remaining nine


## Kaggle evidence

Abhyuday's [RL journey discussion](https://www.kaggle.com/competitions/pokemon-tcg-ai-battle/discussion/717697) emphasizes representation quality, millions of games, a refined curriculum, exposure to both using and beating the important cards, and replay analysis of specific losing situations. His reported best used roughly 3–5 million games, versus 372,736 scheduled source games here.

The [RL/PPO/MCTS discussion](https://www.kaggle.com/competitions/pokemon-tcg-ai-battle/discussion/711644) contains a directly relevant Lucario result: behavior cloning reached 66% action accuracy but only 10% win rate; a mixed self-play/heuristic PPO league reportedly reached 25%. This supports league anchoring and also shows why high action accuracy is not itself strength. A Crustle BC+PPO result in the same thread reportedly gained about 65 Elo over its rule bot.

Abhyuday's [30,000-game methods discussion](https://www.kaggle.com/competitions/pokemon-tcg-ai-battle/discussion/724362) argues that search helps only when the value head can distinguish better states. Search should therefore wait for calibration evidence rather than being used to rescue the current plateau.

## Takeaways

1. Fix the absolute evaluation and promotion contract before using it to select a specialist. Preserve deck identity, rotate deck coverage, keep permanent anchors, and require candidate non-regression on a locked absolute suite before deployment.
2. Run isolated specialist branches rather than overwriting the generalist: Crustle first, then Cornerstone Ogerpon and Rockets Mewtwo; include Alakazam when optimizing broader ladder coverage.
3. Treat Lucario as a behavior-cloning-plus-league control, not the lead branch. Clone the official Lucario heuristic, then fine-tune with a mixed opponent league and a KL anchor.
4. Target the concrete failure buckets (Iono, Dragapult ex, Mega Abomasnow ex) and replay the decisions that lead to those losses.
5. Add bounded search only after the value head passes held-out ranking/calibration tests.